In [1]:
import pandas as pd
import scipy.stats as stats
import numpy as np
import ast

In [2]:
def compute_accuracy_local(df):
    list_Trustworthiness = df["Trustworthiness"].tolist()
    list_Continuity = df["Continuity"].tolist()

    results = []
    for i in range(len(list_Trustworthiness)):
        results.append(round(0.5*list_Trustworthiness[i] + 0.5*list_Continuity[i], 2))
    return results

In [3]:
def compute_accuracy_global(df):
    list_Shephard = df["Shephard Diagram Correlation"].tolist()

    results = []
    for i in range(len(list_Shephard)):
        results.append(round(0.5*(list_Shephard[i] + 1), 2))
    return results

In [4]:
def compute_perception(df):
    list_NeighborhoodHit = df["7-Neighborhood Hit"].tolist()
    list_DistanceConsistency = df["Distance consistency"].tolist()

    results = []
    for i in range(len(list_NeighborhoodHit)):
        results.append(round(0.5*list_NeighborhoodHit[i] + 0.5*list_DistanceConsistency[i], 2))
    return results

In [5]:
def extract_K(df, K):
    # Define excluded embeddings
    excluded_embeddings = ['bert', 'bow', 'tfidf']

    # Keep only TMs (LDA, LSI, NMF) and make a copy to avoid SettingWithCopyWarning
    df_TMs = df[~df['TM'].isin(excluded_embeddings)].copy()

    # Extract the number of topics from the experiment name
    df_TMs['n_topics'] = df_TMs['Experiment'].str.extract(r'n_topics_(\d+)_')[0]

    # Keep only rows with the desired number of topics
    df_TMs = df_TMs[df_TMs['n_topics'] == str(K)]

    # Keep rows with no TMs and make a copy
    df_noTMs = df[df['TM'].isin(excluded_embeddings)].copy()
    df_noTMs['n_topics'] = str(K)

    # Combine the two DataFrames
    df_result = pd.concat([df_TMs, df_noTMs], ignore_index=True)

    return df_result


In [6]:
def process_df(file, corpus, K):
    df = pd.read_csv(file)
    df["corpus"] = corpus
    df["accuracy_local"] = compute_accuracy_local(df)
    df["accuracy_global"] = compute_accuracy_global(df)
    df["perception"] = compute_perception(df)
    df_result = extract_K(df, K)
    return df_result

In [7]:
df_20Newsgroups = process_df("results_cluster/cur_res/full_res_20_newsgroups.csv", "20Newsgroups", K = 20)
print(set(df_20Newsgroups["DR"].tolist()))
print(len(set(df_20Newsgroups["TM"].tolist())))
print(df_20Newsgroups.shape)

{'umap', 'som', 'tsne'}
13
(3767, 19)


In [8]:
df_Emails = process_df("results_cluster/cur_res/full_res_emails.csv", "Emails", K = 8)
print(set(df_Emails["DR"].tolist()))
print(len(set(df_Emails["TM"].tolist())))
print(df_Emails.shape)

{'umap', 'som', 'mds', 'tsne'}
12
(3622, 19)


In [9]:
df_BBC = process_df("results_cluster/cur_res/full_res_bbc_news.csv", "BBC", K = 10)
print(set(df_BBC["DR"].tolist()))
print(len(set(df_BBC["TM"].tolist())))
print(df_BBC.shape)

{'umap', 'som', 'mds', 'tsne'}
13
(3834, 19)


In [10]:
df_lyrics = process_df("results_cluster/cur_res/full_res_lyrics.csv", "Lyrics", K = 8)
print(set(df_lyrics["DR"].tolist()))
print(len(set(df_lyrics["TM"].tolist())))
print(df_lyrics.shape)

{'umap', 'som', 'mds', 'tsne'}
13
(3849, 19)


In [11]:
df_Reuters = process_df("results_cluster/cur_res/full_res_reuters.csv", "Reuters", K = 10)
print(set(df_Reuters["DR"].tolist()))
print(len(set(df_Reuters["TM"].tolist())))
print(df_Reuters.shape)

{'umap', 'som', 'mds', 'tsne'}
13
(3849, 19)


In [12]:
df_7Categories = process_df("results_cluster/cur_res/full_res_seven_categories.csv", "7Categories", K = 14)
print(set(df_7Categories["DR"].tolist()))
print(len(set(df_7Categories["TM"].tolist())))
print(df_7Categories.shape)

{'umap', 'som', 'tsne'}
13
(3754, 19)


In [13]:
df_all_corpora = pd.concat([df_20Newsgroups, df_Emails, df_BBC, df_lyrics, df_Reuters, df_7Categories], ignore_index=True)
df_all_corpora.shape

(22675, 19)

In [14]:
set(df_all_corpora["corpus"].tolist())

{'20Newsgroups', '7Categories', 'BBC', 'Emails', 'Lyrics', 'Reuters'}

In [15]:
columns = ["corpus","embedding","DR","layout-quality-measure","value"]
df_optimums = pd.DataFrame(columns = columns)

corpora_list = ['20Newsgroups', 'Emails', '7Categories', 'BBC', 'Lyrics', 'Reuters']
TM_list = ['bow','tfidf','lda','lda_linear_combined','lsi','lsi_linear_combined','lsi_tfidf','lsi_tfidf_linear_combined','nmf','nmf_linear_combined',
           'nmf_tfidf','nmf_tfidf_linear_combined','bert']
DR_list = ['mds', 'som', 'tsne', 'umap']

for corpus in corpora_list:
    for TM in TM_list:
        for DR in DR_list:
            try:
                max_accloc = max(df_all_corpora[(df_all_corpora["corpus"] == corpus) & (df_all_corpora["TM"] == TM) & (df_all_corpora["DR"] == DR)]["accuracy_local"].tolist())
                max_accglo = max(df_all_corpora[(df_all_corpora["corpus"] == corpus) & (df_all_corpora["TM"] == TM) & (df_all_corpora["DR"] == DR)]["accuracy_global"].tolist())
                max_perception = max(df_all_corpora[(df_all_corpora["corpus"] == corpus) & (df_all_corpora["TM"] == TM) & (df_all_corpora["DR"] == DR)]["perception"].tolist())
            except:
                max_accloc = 0
                max_accglo = 0
                max_perception = 0
            new_row_accloc = {"corpus":corpus,"embedding":TM,"DR":DR,"layout-quality-measure":"accloc","value":max_accloc}
            new_row_accglo = {"corpus":corpus,"embedding":TM,"DR":DR,"layout-quality-measure":"accglo","value":max_accglo}
            new_row_perception = {"corpus":corpus,"embedding":TM,"DR":DR,"layout-quality-measure":"perception","value":max_perception}
            df_optimums = pd.concat([df_optimums, pd.DataFrame([new_row_accloc]), pd.DataFrame([new_row_accglo]), pd.DataFrame([new_row_perception])], ignore_index=True)      

In [16]:
df_optimums[df_optimums["value"] != 0]

,corpus,embedding,DR,layout-quality-measure,value
3,20Newsgroups,bow,som,accloc,0.8
4,20Newsgroups,bow,som,accglo,0.57
5,20Newsgroups,bow,som,perception,0.25
6,20Newsgroups,bow,tsne,accloc,0.93
7,20Newsgroups,bow,tsne,accglo,0.62
...,...,...,...,...,...
931,Reuters,bert,tsne,accglo,0.79
932,Reuters,bert,tsne,perception,0.3
933,Reuters,bert,umap,accloc,0.97
934,Reuters,bert,umap,accglo,0.8


In [ ]:
#df_optimums.to_csv("analysis-results/optimums_dissertation.csv")
df_optimums.to_csv("analysis-results/optimums_dissertation-ueberpruefung.csv")